In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
국회 회의록 데이터 수집 스크립트
- 국회 Open API로 회의록 목록 수집
- HTML 파싱으로 상세 정보 추출
- 정당 정보 자동 매핑 및 추가
- CSV 파일로 저장 (data/{회차명}/ 폴더 구조)
"""

import sys
import os
import re
import time
from pathlib import Path
from typing import List, Optional, Dict, Any

import pandas as pd
import requests
from bs4 import BeautifulSoup, NavigableString, Tag
from dotenv import load_dotenv
import urllib3
import openpyxl

# ---------------------------------------------------------------------------
# 프로젝트 루트에서 .env 파일 로드 (스크립트 + Jupyter 둘 다 동작하도록 수정)
# ---------------------------------------------------------------------------
try:
    # .py 스크립트로 실행되는 경우 (원래 동작)
    project_root = Path(__file__).resolve().parents[1]  # analysis_scripts -> data_collection
except NameError:
    # Jupyter 노트북에서 셀로 실행되는 경우 (__file__ 없음)
    project_root = Path.cwd()

env_path = project_root / ".env"
load_dotenv(dotenv_path=env_path)

# SSL 인증서 검증을 끌 때 나오는 경고 숨김 (내부망/테스트 환경용)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# ============================================================================
# API 클라이언트
# ============================================================================

class AssemblyAPIClient:
    """국회 Open API 클라이언트"""
    
    BASE_URL = "https://open.assembly.go.kr/portal/openapi/ncwgseseafwbuheph"
    
    def __init__(self, api_key: str, page_size: int = 100, request_delay: float = 0.5):
        self.api_key = api_key
        self.page_size = page_size
        self.request_delay = request_delay
        self.headers = {
            "User-Agent": "Mozilla/5.0 (compatible; MOIS-Intern-Project/1.0)"
        }
    
    def search_meetings(
        self,
        dae_num: str,
        conf_date: str,
        comm_name: Optional[str] = None,
        max_pages: Optional[int] = None
    ) -> List[Dict[str, Any]]:
        """위원회 회의록 검색"""
        all_records = []
        page_index = 1
        
        while True:
            params = {
                "KEY": self.api_key,
                "Type": "json",
                "pIndex": page_index,
                "pSize": self.page_size,
                "DAE_NUM": dae_num,
                "CONF_DATE": conf_date
            }
            
            if comm_name:
                params["COMM_NAME"] = comm_name
            
            try:
                response = requests.get(
                    self.BASE_URL,
                    params=params,
                    headers=self.headers,
                    timeout=30,
                    verify=False,  # SSL 인증서 검증 비활성화
                )
                response.raise_for_status()
                data = response.json()
                
                # 응답 파싱
                records = self._parse_response(data)
                
                if not records:
                    break
                
                all_records.extend(records)
                
                if max_pages and page_index >= max_pages:
                    break
                
                if len(records) < self.page_size:
                    break
                
                page_index += 1
                time.sleep(self.request_delay)
            
            except Exception as e:
                print(f"  ❌ 페이지 {page_index} 처리 중 오류: {e}")
                break
        
        return all_records
    
    def _parse_response(self, data: Dict[str, Any]) -> List[Dict[str, Any]]:
        """API 응답 파싱"""
        records = []
        
        if not isinstance(data, dict):
            return records
        
        # API 응답의 루트 키 찾기
        root_key = None
        for key in data.keys():
            if "ncwgseseafwbuheph" in key.lower():
                root_key = key
                break
        
        if not root_key:
            if "row" in data:
                root_data = data
            else:
                return records
        else:
            root_data = data[root_key]
        
        # row 데이터 추출
        rows = []
        if isinstance(root_data, list):
            for item in root_data:
                if isinstance(item, dict) and "row" in item:
                    if isinstance(item["row"], list):
                        rows.extend(item["row"])
                    else:
                        rows.append(item["row"])
        elif isinstance(root_data, dict):
            if "row" in root_data:
                if isinstance(root_data["row"], list):
                    rows = root_data["row"]
                else:
                    rows = [root_data["row"]]
        
        return rows


# ============================================================================
# HTML 파서
# ============================================================================

HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; MOIS-Intern-Project/1.0)"}
SLEEP_SEC = 0.6


def clean(s: str) -> str:
    """텍스트 정리"""
    if not s:
        return ""
    s = s.replace("\xa0", " ")
    s = re.sub(r"\s+", " ", s)
    return s.strip()


def get_soup(url: str) -> BeautifulSoup:
    """HTML/XML 뷰어 URL에서 Soup 객체 생성"""
    try:
        r = requests.get(url, headers=HEADERS, timeout=25, verify=False)  # SSL 인증서 검증 비활성화
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")
        return soup
    except Exception as e:
        print(f"  ❌ request error: {e}")
        raise


def parse_header_title(soup: BeautifulSoup):
    """회의록 헤더 정보 파싱"""
    session, session_type, meeting_no = None, "", ""
    tit_wrap = soup.select_one("#minutes .minutes_header .tit_wrap")
    if tit_wrap:
        turn = tit_wrap.select_one("p.turn")
        if turn:
            t = turn.get_text("\n", strip=True)
            m1 = re.search(r"제\s*(\d+)\s*회", t)
            if m1:
                session = int(m1.group(1))
            m2 = re.search(r"\(([^)]+)\)", t)
            if m2:
                session_type = clean(m2.group(1))
        pnum = tit_wrap.select_one("p.num")
        if pnum:
            meeting_no = clean(pnum.get_text())
        if not meeting_no:
            header_text = clean(soup.select_one("#minutes .minutes_header").get_text(" ")) \
                          if soup.select_one("#minutes .minutes_header") else ""
            m3 = re.search(r"제\s*\d+\s*[차호]", header_text)
            if m3:
                meeting_no = m3.group(0)
    return session, session_type, meeting_no


def parse_date_only(soup: BeautifulSoup):
    """날짜 파싱"""
    date_text = ""
    for li in soup.select("#minutes .minutes_header .place ul > li"):
        sbj_el = li.select_one(".sbj, .sbj.lts2, .sbj.lts4")
        sbj = clean(sbj_el.get_text()) if sbj_el else ""
        if "일시" in sbj:
            con_el = li.select_one("p.con")
            if con_el:
                date_text = clean(con_el.get_text())
            break
    return date_text


def get_text_with_br(container: Tag) -> str:
    """BR 태그를 줄바꿈으로 변환"""
    if container is None:
        return ""
    parts = []
    for node in container.descendants:
        if isinstance(node, NavigableString):
            parts.append(str(node))
        elif isinstance(node, Tag) and node.name.lower() == "br":
            parts.append("\n")
    text = "".join(parts)
    text = text.replace("\xa0", " ")
    text = re.sub(r"\n\s*\n+", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()


def _parse_agenda_title(a_or_p: Tag):
    """안건 제목 파싱"""
    txt = ""
    if a_or_p:
        txt = clean(a_or_p.get_text())
    m = re.match(r"(\d+)\.\s*(.+)", txt)
    if m:
        return m.group(1), clean(m.group(2))
    return "", txt


def _nearest_time_after(node: Tag):
    """가장 가까운 시간 찾기"""
    cur = node
    for _ in range(12):
        cur = cur.find_next()
        if not isinstance(cur, Tag):
            continue
        cls = cur.get("class", [])
        if cur.name == "p" and "tit_sm" in cls and "taR" in cls:
            return clean(cur.get_text())
        if cur.name == "p" and "tit_sm" in cls and "angun" in cls:
            break
        if cur.name == "div" and "speaker" in cls:
            break
    return ""


def _extract_agenda_groups(soup: BeautifulSoup):
    """안건 그룹 추출"""
    speakers = soup.select("#minutes .minutes_body .speaker")
    bars = soup.select("#minutes .minutes_body p.tit_sm.angun")
    
    groups = []
    i = 0
    while i < len(bars):
        cur_bar = bars[i]
        orders, titles, times = [], [], []
        
        while True:
            a_tit = cur_bar.select_one("a.tit")
            num, title = _parse_agenda_title(a_tit if a_tit else cur_bar)
            if num:
                orders.append(num)
            if title:
                titles.append(title)
            tm = _nearest_time_after(cur_bar)
            if tm:
                times.append(tm)
            
            next_bar = bars[i+1] if i+1 < len(bars) else None
            if not next_bar:
                break
            found_speaker_between = False
            probe = cur_bar
            while True:
                probe = probe.find_next()
                if not probe or probe is next_bar:
                    break
                if isinstance(probe, Tag) and "speaker" in probe.get("class", []):
                    found_speaker_between = True
                    break
            if found_speaker_between:
                break
            i += 1
            cur_bar = next_bar
        
        first_spk_after_group = cur_bar.find_next(class_="speaker")
        start_order = None
        if first_spk_after_group:
            for idx, spk in enumerate(soup.select("#minutes .minutes_body .speaker"), start=1):
                if spk is first_spk_after_group:
                    start_order = idx
                    break
        if start_order is not None:
            groups.append({
                "start_order": start_order,
                "orders": ",".join(orders) if orders else "",
                "titles": " / ".join(titles) if titles else "",
                "times": " / ".join(times) if times else "",
            })
        
        i += 1
    
    groups.sort(key=lambda g: g["start_order"])
    return groups, soup.select("#minutes .minutes_body .speaker")


def parse_header(url: str):
    """헤더 파싱"""
    soup = get_soup(url)
    session, session_type, meeting_no = parse_header_title(soup)
    date_text = parse_date_only(soup)
    
    rows = []
    rows.append({
        "session": session, "session_type": session_type, "meeting_no": meeting_no,
        "date": date_text, "section": "일시", "item_order": "", "item_text": date_text,
    })
    
    for li in soup.select("#minutes .minutes_header .place ul > li"):
        sbj_el = li.select_one(".sbj, .sbj.lts4, .sbj.lts2")
        sbj = clean(sbj_el.get_text()) if sbj_el else ""
        if "의사일정" in sbj or "상정된 안건" in sbj:
            label = "의사일정" if "의사일정" in sbj else "상정된 안건"
            items = li.select("ul.list_num li")
            for idx, it in enumerate(items, start=1):
                a = it.select_one("a")
                txt = clean(a.get_text()) if a and clean(a.get_text()) else clean(it.get_text())
                txt = re.sub(r"^\d+\.\s*", "", txt)
                rows.append({
                    "session": session, "session_type": session_type, "meeting_no": meeting_no,
                    "date": date_text, "section": label, "item_order": idx, "item_text": txt,
                })
    
    return rows, (session, meeting_no)


def parse_speeches(url: str):
    """발언 파싱"""
    soup = get_soup(url)
    session, session_type, meeting_no = parse_header_title(soup)
    date_text = parse_date_only(soup)
    
    groups, speakers = _extract_agenda_groups(soup)
    
    ranges = []
    for idx, g in enumerate(groups):
        start = g["start_order"]
        end = (groups[idx+1]["start_order"] - 1) if idx+1 < len(groups) else len(speakers)
        ranges.append((start, end, g))
    
    rows = []
    for order, spk in enumerate(speakers, start=1):
        data_mem_id = ""
        for key in ("data-mem_id", "data_mem_id", "data-mem-id"):
            if spk.has_attr(key):
                data_mem_id = spk.get(key) or ""
                break
        
        name_el = spk.select_one(".man .txt strong.name")
        pos_el = spk.select_one(".man .txt .position")
        area_el = spk.select_one(".man .txt .area")
        
        name = clean(name_el.get_text()) if name_el else clean(spk.get("data-name", ""))
        position = clean(pos_el.get_text()) if pos_el else clean(spk.get("data-pos", ""))
        area = clean(area_el.get_text()) if area_el else ""
        
        speech = get_text_with_br(spk.select_one(".talk .txt"))
        
        ag_orders = ag_titles = ag_times = ""
        for start, end, g in ranges:
            if start <= order <= end:
                ag_orders = g["orders"]
                ag_titles = g["titles"]
                ag_times = g["times"]
                break
        
        rows.append({
            "session": session,
            "session_type": session_type,
            "meeting_no": meeting_no,
            "date": date_text,
            "speech_order": order,
            "speaker_name": name,
            "speaker_position": position,
            "speaker_area": area,
            "data_mem_id": data_mem_id,
            "agenda_item_orders": ag_orders,
            "agenda_item_titles": ag_titles,
            "agenda_item_times": ag_times,
            "speech_text": speech,
        })
    return rows, (session, meeting_no)


def safe_name(s: str) -> str:
    """파일명 안전화"""
    s = s.replace("/", "_").replace("\\", "_").replace(":", "_")
    s = s.replace("?", "").replace("*", "").replace('"', "").replace("<", "").replace(">", "").replace("|", "")
    return s.strip()


# ============================================================================
# 정당 정보 매핑
# ============================================================================

PARTY_MAPPING = {
 # 20대
    '강길부' : '더불어민주당', '강병원' : '더불어민주당', '강석진' : '새누리당', '강석호' : '새누리당', '강창일' : '더불어민주당',
    '강효상' : '새누리당', '강훈식' : '더불어민주당', '경대수' : '국민의당', '고용진' : '더불어민주당', '곽대훈' : '국민의당',
    '곽상도' : '새누리당', '권미혁' : '더불어민주당', '권석창' : '국민의당', '권성동' : '새누리당', '권은희' : '국민의당',
    '권칠승' : '더불어민주당', '금태섭' : '더불어민주당', '기동민' : '더불어민주당', '김경수' : '더불어민주당', '김경진' : '국민의당',
    '김경협' : '더불어민주당', '김관영' : '국민의당', '김광림' : '새누리당', '김광수' : '국민의당', '김규환' : '새누리당',
    '김기선' : '새누리당', '김도읍' : '새누리당', '김동철' : '국민의당', '김두관' : '더불어민주당', '김명연' : '새누리당',
    '김무성' : '새누리당', '김민기' : '더불어민주당', '김병관' : '더불어민주당', '김병기' : '더불어민주당', '김병욱' : '더불어민주당',
    '김부겸' : '더불어민주당', '김삼화' : '국민의당', '김상훈' : '새누리당', '김상희' : '더불어민주당', '김석기' : '새누리당',
    '김선동' : '국민의당', '김성수' : '국민의당', '김성식' : '새누리당', '김성원' : '새누리당', '김성찬' : '새누리당',
    '김성태(金成泰)' : '새누리당', '김성태(金聖泰)' : '새누리당', '김성환' : '더불어민주당', '김세연' : '새누리당', '김수민' : '국민의당',
    '김순례' : '새누리당', '김승희' : '새누리당', '김영우' : '새누리당', '김영주' : '더불어민주당', '김영진' : '국민의당',
    '김영춘' : '더불어민주당', '김영호' : '새누리당', '김용태' : '새누리당', '김재경' : '새누리당', '김재원' : '새누리당',
    '김정우' : '더불어민주당', '김정재' : '새누리당', '김정호' : '국민의당', '김정훈' : '새누리당', '김종대' : '정의당',
    '김종민' : '더불어민주당', '김종석' : '국민의당', '김종인' : '더불어민주당', '김종태' : '국민의당', '김종회' : '국민의당',
    '김종훈' : '국민의당', '김중로' : '새누리당', '김진태' : '새누리당', '김진표' : '더불어민주당', '김철민' : '더불어민주당',
    '김태년' : '더불어민주당', '김태흠' : '새누리당', '김학용' : '새누리당', '김한정' : '더불어민주당', '김한표' : '새누리당',
    '김해영' : '더불어민주당', '김현권' : '더불어민주당', '김현미' : '더불어민주당', '김현아' : '새누리당', '나경원' : '새누리당',
    '노웅래' : '더불어민주당', '류성걸' : '새누리당', '민경욱' : '새누리당', '민병두' : '더불어민주당', '민홍철' : '더불어민주당',
    '박경미' : '더불어민주당', '박광온' : '더불어민주당', '박남춘' : '더불어민주당', '박대출' : '새누리당', '박덕흠' : '새누리당',
    '박맹우' : '새누리당', '박명재' : '새누리당', '박범계' : '더불어민주당', '박병석' : '더불어민주당', '박선숙' : '국민의당',
    '박성중' : '새누리당', '박순자' : '새누리당', '박영선' : '더불어민주당', '박완수' : '새누리당', '박완주' : '더불어민주당',
    '박용진' : '더불어민주당', '박인숙' : '새누리당', '박재호' : '더불어민주당', '박정' : '더불어민주당', '박주민' : '더불어민주당',
    '박주선' : '국민의당', '박주현' : '국민의당', '박준영' : '국민의당', '박지원' : '국민의당', '박찬대' : '더불어민주당',
    '박찬우' : '국민의당', '박홍근' : '더불어민주당', '배덕광' : '새누리당', '백승주' : '새누리당', '백재현' : '더불어민주당',
    '백혜련' : '더불어민주당', '변재일' : '더불어민주당', '서삼석' : '더불어민주당', '서영교' : '더불어민주당', '서청원' : '새누리당',
    '서형수' : '새누리당', '설훈' : '더불어민주당', '성일종' : '새누리당', '소병훈' : '더불어민주당', '손금주' : '국민의당',
    '손혜원' : '더불어민주당', '송갑석' : '더불어민주당', '송기석' : '국민의당', '송기헌' : '더불어민주당', '송석준' : '새누리당',
    '송언석' : '새누리당', '송영길' : '더불어민주당', '송옥주' : '더불어민주당', '송희경' : '새누리당', '신경민' : '더불어민주당',
    '신동근' : '더불어민주당', '신보라' : '새누리당', '신상진' : '새누리당', '신용현' : '새누리당', '신창현' : '새누리당',
    '심기준' : '새누리당', '심상정' : '정의당', '심재권' : '새누리당', '심재철' : '새누리당', '안규백' : '더불어민주당',
    '안민석' : '더불어민주당', '안상수' : '새누리당', '안철수' : '국민의당', '안호영' : '더불어민주당', '양승조' : '더불어민주당',
    '어기구' : '새누리당', '엄용수' : '새누리당', '여상규' : '새누리당', '여영국' : '새누리당', '염동열' : '새누리당',
    '오세정' : '국민의당', '오신환' : '새누리당', '오영훈' : '더불어민주당', '오제세' : '정의당', '우상호' : '더불어민주당',
    '우원식' : '더불어민주당', '원유철' : '새누리당', '원혜영' : '더불어민주당', '위성곤' : '더불어민주당', '유기준' : '새누리당',
    '유동수' : '국민의당', '유민봉' : '국민의당', '유성엽' : '국민의당', '유승민' : '새누리당', '유승희' : '더불어민주당',
    '유은혜' : '더불어민주당', '유의동' : '새누리당', '유재중' : '국민의당', '윤관석' : '더불어민주당', '윤상직' : '새누리당',
    '윤상현' : '새누리당', '윤소하' : '정의당', '윤영석' : '새누리당', '윤영일' : '새누리당', '윤일규' : '국민의당',
    '윤재옥' : '새누리당', '윤종오' : '새누리당', '윤종필' : '새누리당', '윤준호' : '국민의당', '윤한홍' : '새누리당',
    '윤호중' : '더불어민주당', '윤후덕' : '더불어민주당', '이개호' : '더불어민주당', '이군현' : '새누리당', '이규희' : '국민의당',
    '이동섭' : '새누리당', '이만희' : '새누리당', '이명수' : '새누리당', '이상돈' : '국민의당', '이상민' : '더불어민주당',
    '이상헌' : '국민의당', '이석현' : '더불어민주당', '이수혁' : '국민의당', '이양수' : '새누리당', '이언주' : '더불어민주당',
    '이완영' : '새누리당', '이용득' : '더불어민주당', '이용주' : '국민의당', '이용호' : '더불어민주당', '이우현' : '새누리당',
    '이원욱' : '더불어민주당', '이은권' : '새누리당', '이은재' : '새누리당', '이인영' : '더불어민주당', '이장우' : '새누리당',
    '이재정' : '더불어민주당', '이정미' : '정의당', '이정현' : '새누리당', '이종걸' : '더불어민주당', '이종구' : '새누리당',
    '이종명' : '새누리당', '이종배' : '새누리당', '이주영' : '새누리당', '이진복' : '새누리당', '이찬열' : '국민의당',
    '이채익' : '새누리당', '이철규' : '새누리당', '이철우' : '새누리당', '이철희' : '더불어민주당', '이춘석' : '더불어민주당',
    '이태규' : '새누리당', '이학영' : '더불어민주당', '이학재' : '새누리당', '이해찬' : '더불어민주당', '이헌승' : '새누리당',
    '이현재' : '새누리당', '이혜훈' : '새누리당', '이후삼' : '새누리당', '이훈' : '국민의당', '인재근' : '더불어민주당',
    '임이자' : '새누리당', '임재훈' : '새누리당', '임종성' : '더불어민주당', '장병완' : '국민의당', '장석춘' : '국민의당',
    '장정숙' : '더불어민주당', '장제원' : '새누리당', '전재수' : '새누리당', '전해철' : '더불어민주당', '전현희' : '더불어민주당',
    '전혜숙' : '더불어민주당', '전희경' : '새누리당', '정갑윤' : '새누리당', '정동영' : '국민의당', '정병국' : '새누리당',
    '정성호' : '더불어민주당', '정세균' : '더불어민주당', '정양석' : '국민의당', '정용기' : '새누리당', '정우택' : '새누리당',
    '정운천' : '새누리당', '정유섭' : '새누리당', '정은혜' : '국민의당', '정인화' : '국민의당', '정점식' : '국민의당',
    '정재호' : '더불어민주당', '정종섭' : '새누리당', '정진석' : '새누리당', '정춘숙' : '더불어민주당', '정태옥' : '새누리당',
    '제윤경' : '새누리당', '조경태' : '새누리당', '조배숙' : '더불어민주당', '조승래' : '더불어민주당', '조원진' : '새누리당',
    '조응천' : '더불어민주당', '조정식' : '더불어민주당', '조훈현' : '새누리당', '주광덕' : '새누리당', '주승용' : '국민의당',
    '주호영' : '새누리당', '지상욱' : '새누리당', '진선미' : '더불어민주당', '진영' : '더불어민주당', '채이배' : '국민의당',
    '천정배' : '국민의당', '최경환(崔炅煥)' : '새누리당', '최경환(崔敬煥)' : '국민의당', '최교일' : '더불어민주당',
    '최도자' : '국민의당', '최명길' : '새누리당', '최연혜' : '더불어민주당', '최운열' : '국민의당', '최인호' : '새누리당',
    '최재성' : '더불어민주당', '추경호' : '새누리당', '추미애' : '더불어민주당', '추혜선' : '정의당', '하태경' : '새누리당',
    '한선교' : '새누리당', '한정애' : '더불어민주당', '함진규' : '새누리당', '허윤정' : '국민의당', '홍문종' : '새누리당',
    '홍문표' : '새누리당', '홍영표' : '더불어민주당', '홍의락' : '국민의당', '홍익표' : '더불어민주당', '홍일표' : '새누리당',
    '홍철호' : '국민의당', '황영철' : '새누리당', '황주홍' : '국민의당', '황희' : '더불어민주당'

}


def add_party_to_speeches(speech_rows):
    """발언 데이터에 정당 정보 추가"""
    if not speech_rows:
        return speech_rows
    
    df = pd.DataFrame(speech_rows)
    df['party'] = df['speaker_name'].map(PARTY_MAPPING).fillna('미분류')
    return df.to_dict('records')


# ============================================================================
# 메인 수집 함수
# ============================================================================

def collect_data(session_name: Optional[str] = None, limit: Optional[int] = None):
    """
    데이터 수집 프로세스 실행
    
    Args:
        session_name: 회차명 (None이면 모든 회차 처리)
        limit: 처리할 회의록 개수 (None이면 전체)
    """
    print(f"\n{'='*70}")
    print("📊 국회 회의록 데이터 수집")
    print(f"{'='*70}")
    
    # API 키 확인
    api_key = os.getenv("ASSEMBLY_API_KEY")
    if not api_key:
        print("  ❌ ASSEMBLY_API_KEY 환경 변수가 설정되지 않았습니다.")
        print("  .env 파일에 ASSEMBLY_API_KEY를 추가해주세요.")
        return
    
    api_client = AssemblyAPIClient(api_key=api_key, page_size=100)
    
    dae_num = "20"
    years = ["2016", "2017", "2018", "2019", "2020"]
    
    if session_name:
        match = re.search(r"제\s*(\d+)\s*회", session_name)
        if not match:
            print(f"  ❌ 잘못된 회차명: {session_name}")
            return
    
    # API 호출
    print(f"\n[단계 1] API 호출 중... (대수: {dae_num}, 연도: {', '.join(years)})")
    
    all_records = []
    for year in years:
        year_records = api_client.search_meetings(
            dae_num=dae_num,
            conf_date=year,
            comm_name="행정안전위원회",
            max_pages=50
        )
        all_records.extend(year_records)
        print(f"  ✓ {year}년: {len(year_records)}개 수집")
    
    records = all_records
    print(f"  ✅ 총 {len(records)}개 회의록 수집 완료")
    
    # 회차 분류
    print(f"\n[단계 2] 회차 분류 중...")
    
    if session_name:
        session_records = []
        for record in records:
            if session_name in record.get("TITLE", ""):
                session_records.append(record)
        
        if not session_records:
            print(f"  ⚠️  {session_name}에 해당하는 회의록이 없습니다.")
            return
        
        sessions_dict = {session_name: session_records}
        print(f"  ✓ {session_name}: {len(session_records)}개 회의록")
    else:
        # 모든 레코드에서 회차 추출
        sessions_dict = {}
        for record in records:
            title = record.get("TITLE", "")
            match = re.search(r"제\s*(\d+)\s*회", title)
            if match:
                session_key = f"제{match.group(1)}회"
                if session_key not in sessions_dict:
                    sessions_dict[session_key] = []
                sessions_dict[session_key].append(record)
        
        print(f"  ✓ 발견된 회차: {len(sessions_dict)}개")
        for session_key in sorted(sessions_dict.keys(), key=lambda x: int(re.search(r"제\s*(\d+)\s*회", x).group(1))):
            print(f"    - {session_key}: {len(sessions_dict[session_key])}개")
        
        if not sessions_dict:
            print(f"  ⚠️  회차를 찾을 수 없습니다.")
            return
    
    # 데이터 처리
    base_outdir = project_root / "data"
    total_created = 0
    total_failed = 0
    
    print(f"\n[단계 3] 데이터 처리 시작...")
    
    for session_key, session_records in sessions_dict.items():
        # 중복 제거
        meetings_by_id = {}
        for record in session_records:
            confer_num = record.get("CONFER_NUM", "")
            if confer_num and confer_num not in meetings_by_id:
                meetings_by_id[confer_num] = record
        
        # 제한 적용
        limited_meetings = dict(list(meetings_by_id.items())[:limit]) if limit else meetings_by_id
        
        # 회차별 폴더 생성
        session_dir = base_outdir / session_key
        session_dir.mkdir(parents=True, exist_ok=True)
        
        print(f"\n  📁 {session_key}: {len(limited_meetings)}개 회의록 처리 중...")
        
        session_created = 0
        session_failed = 0
        
        for idx, (confer_num, main_record) in enumerate(limited_meetings.items(), 1):
            try:
                # HTML URL 생성 및 파싱
                html_url = f"https://record.assembly.go.kr/assembly/viewer/minutes/xml.do?id={confer_num}&type=view"
                
                header_rows, (parsed_session, parsed_meeting_no) = parse_header(html_url)
                
                if not parsed_session:
                    if session_name:
                        parsed_session = int(re.search(r"제\s*(\d+)\s*회", session_name).group(1))
                    else:
                        parsed_session = int(re.search(r"제\s*(\d+)\s*회", session_key).group(1))
                if not parsed_meeting_no:
                    parsed_meeting_no = f"제{confer_num}호"
                
                speech_rows, _ = parse_speeches(html_url)
                
                # 정당 정보 추가
                if speech_rows:
                    speech_rows = add_party_to_speeches(speech_rows)
                
                fname_prefix = safe_name(parsed_meeting_no)
                # # CSV 저장
                # header_path = session_dir / f"{fname_prefix}_minutes_header_summary.csv"
                # speech_path = session_dir / f"{fname_prefix}_minutes_speeches.csv"
                
                # pd.DataFrame(header_rows).to_csv(header_path, index=False, encoding="utf-8-sig")
                # if speech_rows:
                #     pd.DataFrame(speech_rows).to_csv(speech_path, index=False, encoding="utf-8-sig")


                # ===== 엑셀 저장 (원하면 아래 주석 해제해서 사용) =====
                header_path_xlsx = session_dir / f"{fname_prefix}_minutes_header_summary.xlsx"
                speech_path_xlsx = session_dir / f"{fname_prefix}_minutes_speeches.xlsx"
                
                pd.DataFrame(header_rows).to_excel(header_path_xlsx, index=False)
                if speech_rows:
                    pd.DataFrame(speech_rows).to_excel(speech_path_xlsx, index=False)
                
                session_created += 1

                if not has_speech:
                    print(f"      ⚠ 발언 데이터 없음 → header만 저장 (CONFER_NUM={confer_num})")
                
                # 진행 상황 출력 (10개마다 또는 마지막)
                if idx % 10 == 0 or idx == len(limited_meetings):
                    print(f"    [{idx}/{len(limited_meetings)}] 처리 완료... (성공: {session_created}, 실패: {session_failed})")
                
                time.sleep(SLEEP_SEC)
                
            except Exception as e:
                session_failed += 1
                if idx % 10 == 0 or idx == len(limited_meetings):
                    print(f"    [{idx}/{len(limited_meetings)}] 처리 중... (실패: {session_failed})")
                continue
        
        total_created += session_created
        total_failed += session_failed
        
        print(f"  ✓ {session_key} 완료: 성공 {session_created}개, 실패 {session_failed}개")
    
    # 완료 요약
    print(f"\n{'='*70}")
    print("✅ 수집 완료")
    print(f"{'='*70}")
    print(f"  총 처리 회차: {len(sessions_dict)}개")
    print(f"  총 성공: {total_created}개")
    print(f"  총 실패: {total_failed}개")
    print(f"  저장 위치: {base_outdir.absolute()}")
    
    # if base_outdir.exists():
    #     session_dirs = [d for d in os.listdir(base_outdir) if os.path.isdir(base_outdir / d) and d != "demo_output" and d != "collection_log.txt"]
    #     print(f"\n  생성된 회차 폴더: {len(session_dirs)}개")
    #     for session_dir_name in sorted(session_dirs):
    #         session_path = base_outdir / session_dir_name
    #         file_count = len([f for f in os.listdir(session_path) if f.endswith('.csv')]) // 2
    #         print(f"    - {session_dir_name}: {file_count}개 회의록")

    if base_outdir.exists():
        session_dirs = [d for d in os.listdir(base_outdir)
                        if os.path.isdir(base_outdir / d)
                        and d != "demo_output"
                        and d != "collection_log.txt"]
        print(f"\n  생성된 회차 폴더: {len(session_dirs)}개")
    
        for session_dir_name in sorted(session_dirs):
            session_path = base_outdir / session_dir_name
    
            xlsx_files = [f for f in os.listdir(session_path)
                          if f.endswith('.xlsx')]
    
            # 파일명 prefix 기준으로 "회의록 개수" 계산
            # 예: "제1차_minutes_header_summary.xlsx" → "제1차"
            meeting_ids = set()
            for fname in xlsx_files:
                if fname.endswith("_minutes_header_summary.xlsx"):
                    meeting_ids.add(fname.replace("_minutes_header_summary.xlsx", ""))
                elif fname.endswith("_minutes_speeches.xlsx"):
                    meeting_ids.add(fname.replace("_minutes_speeches.xlsx", ""))
    
            file_count = len(meeting_ids)
    
            print(f"    - {session_dir_name}: {file_count}개 회의록")


    print(f"\n{'='*70}")

# ============================================================================
# CLI 실행용 (터미널에서 python collect_data.py 로 돌릴 때만)
# 노트북에서 실행할 때는 ipykernel 환경이면 이 블록을 건너뛰도록 처리
# ============================================================================

if __name__ == "__main__" and not any("ipykernel" in arg for arg in sys.argv):
    import argparse
    
    parser = argparse.ArgumentParser(description="국회 회의록 데이터 수집")
    parser.add_argument(
        "--session",
        type=str,
        default=None,
        help="수집할 회차 (예: 제418회). 지정하지 않으면 모든 회차 자동 처리"
    )
    parser.add_argument(
        "--limit",
        type=int,
        default=None,
        help="처리할 회의록 개수 (지정하지 않으면 전체)"
    )
    args = parser.parse_args()
    collect_data(args.session, args.limit)

In [2]:
# 전체 회차 전부
collect_data()

# # 특정 회차만 (예: 제415회)
# collect_data(session_name="제415회")

# # # 테스트로 앞 3개 회의록만
# collect_data(limit=3)




📊 국회 회의록 데이터 수집

[단계 1] API 호출 중... (대수: 20, 연도: 2016, 2017, 2018, 2019, 2020)
  ✓ 2016년: 0개 수집
  ✓ 2017년: 846개 수집
  ✓ 2018년: 1338개 수집
  ✓ 2019년: 2239개 수집
  ✓ 2020년: 218개 수집
  ✅ 총 4641개 회의록 수집 완료

[단계 2] 회차 분류 중...
  ✓ 발견된 회차: 17개
    - 제353회: 82개
    - 제354회: 756개
    - 제355회: 29개
    - 제356회: 264개
    - 제358회: 12개
    - 제360회: 18개
    - 제362회: 18개
    - 제363회: 405개
    - 제364회: 583개
    - 제365회: 18개
    - 제367회: 408개
    - 제368회: 61개
    - 제369회: 345개
    - 제370회: 81개
    - 제371회: 1343개
    - 제376회: 64개
    - 제377회: 154개

[단계 3] 데이터 처리 시작...

  📁 제355회: 5개 회의록 처리 중...
    [5/5] 처리 중... (실패: 5)
  ✓ 제355회 완료: 성공 5개, 실패 5개

  📁 제354회: 15개 회의록 처리 중...
  ❌ request error: 500 Server Error: Internal Server Error for url: https://record.assembly.go.kr/assembly/viewer/minutes/xml.do?id=42458&type=view
  ❌ request error: 500 Server Error: Internal Server Error for url: https://record.assembly.go.kr/assembly/viewer/minutes/xml.do?id=42457&type=view
    [10/15] 처리 중... (실패: 10)
  ❌ request erro

In [3]:
from pathlib import Path

# 이 노트북이 있는 위치 기준으로 data 폴더 지정
# 예: 현재 경로가 .../analysis_scripts 라고 가정하면
base_dir = Path.cwd() / "data"

print("기준 폴더:", base_dir)

if not base_dir.exists():
    raise FileNotFoundError(f"{base_dir} 경로에 data 폴더가 없습니다. 경로를 확인해주세요.")

# demo_output, collection_log.txt 같은 건 제외하고 회차 폴더만 대상으로
session_dirs = sorted([
    d for d in base_dir.iterdir()
    if d.is_dir() and d.name not in ("demo_output", "collection_log.txt")
])

total_meetings = 0

for session_dir in session_dirs:
    # 해당 회차 폴더 안의 모든 xlsx 파일
    xlsx_files = list(session_dir.glob("*.xlsx"))

    # 회의록 ID(prefix) 기준으로 unique 개수 세기
    meeting_ids = set()
    for path in xlsx_files:
        fname = path.name
        if fname.endswith("_minutes_header_summary.xlsx"):
            meeting_ids.add(fname.replace("_minutes_header_summary.xlsx", ""))
        elif fname.endswith("_minutes_speeches.xlsx"):
            meeting_ids.add(fname.replace("_minutes_speeches.xlsx", ""))

    count = len(meeting_ids)
    total_meetings += count

    print(f"- {session_dir.name}: {count}개 회의록 (xlsx 파일 {len(xlsx_files)}개)")

print("\n총 회의록 수:", total_meetings)

기준 폴더: C:\Users\user\Desktop\코드\데이터수집\data
- 제353회: 2개 회의록 (xlsx 파일 4개)
- 제354회: 7개 회의록 (xlsx 파일 14개)
- 제355회: 4개 회의록 (xlsx 파일 8개)
- 제356회: 3개 회의록 (xlsx 파일 6개)
- 제358회: 1개 회의록 (xlsx 파일 2개)
- 제360회: 1개 회의록 (xlsx 파일 2개)
- 제362회: 4개 회의록 (xlsx 파일 8개)
- 제363회: 3개 회의록 (xlsx 파일 6개)
- 제364회: 10개 회의록 (xlsx 파일 20개)
- 제365회: 2개 회의록 (xlsx 파일 4개)
- 제367회: 8개 회의록 (xlsx 파일 16개)
- 제368회: 2개 회의록 (xlsx 파일 4개)
- 제369회: 4개 회의록 (xlsx 파일 8개)
- 제370회: 2개 회의록 (xlsx 파일 4개)
- 제371회: 13개 회의록 (xlsx 파일 26개)
- 제376회: 5개 회의록 (xlsx 파일 10개)
- 제377회: 4개 회의록 (xlsx 파일 8개)
- 제379회: 1개 회의록 (xlsx 파일 2개)
- 제380회: 4개 회의록 (xlsx 파일 8개)
- 제381회: 2개 회의록 (xlsx 파일 4개)
- 제382회: 19개 회의록 (xlsx 파일 38개)
- 제383회: 3개 회의록 (xlsx 파일 6개)
- 제384회: 4개 회의록 (xlsx 파일 8개)
- 제385회: 3개 회의록 (xlsx 파일 6개)
- 제386회: 2개 회의록 (xlsx 파일 4개)
- 제387회: 0개 회의록 (xlsx 파일 0개)
- 제388회: 4개 회의록 (xlsx 파일 8개)
- 제389회: 3개 회의록 (xlsx 파일 6개)
- 제390회: 4개 회의록 (xlsx 파일 8개)
- 제391회: 10개 회의록 (xlsx 파일 20개)
- 제392회: 2개 회의록 (xlsx 파일 4개)
- 제393회: 3개 회의록 (xlsx 파일 6개)
- 제394회: 1개 회의록 (x